# Huấn luyện Decision Tree — Camera AI ONT

Đầu vào: `ids_dataset_ready.csv` + `preprocessing_meta.json` (từ `preprocessing_data_balance.ipynb`).

Kết quả được ghi vào **hai thư mục tách theo mục đích sử dụng**:

| Thư mục | Chứa gì | Dùng khi nào |
|---|---|---|
| **`model_export/`** | Mô hình ONNX, mô hình joblib, hợp đồng vào/ra | Đem đi triển khai — chỉ cần copy đúng thư mục này |
| **`training_results/`** | Biểu đồ, ma trận nhầm lẫn, độ quan trọng feature, báo cáo | Đánh giá chất lượng, viết báo cáo, đối chiếu giữa các lần train |

Tách hai thư mục để lúc triển khai không phải lọc file: mọi thứ trong `model_export/` đều cần cho
lúc chạy, không thứ nào thừa; còn `training_results/` là tài liệu, không đụng tới khi chạy thật.

| Bước | Nội dung |
|---|---|
| 1 | Chuẩn bị môi trường, tạo hai thư mục kết quả |
| 2 | Nạp dữ liệu, khôi phục ba tập train / val / test |
| 3 | Mô hình cơ sở để có mốc so sánh |
| 4 | Dò siêu tham số **trên tập val** |
| 5 | Huấn luyện mô hình cuối |
| 6 | Đánh giá trên tập test |
| 7 | **Ma trận nhầm lẫn** → ảnh + CSV |
| 8 | **% ảnh hưởng của từng feature** → ảnh + CSV |
| 9 | Kiểm tra rò rỉ: đọc luật của cây |
| 10 | Đường ROC / Precision-Recall |
| 11 | Đối chiếu với Random Forest |
| 12 | Lưu mô hình và tổng hợp |
| 13 | Xuất ONNX (`input` / `output`) và liệt kê hai thư mục |

> **Nguyên tắc dùng tập test:** siêu tham số được chọn **chỉ dựa trên tập val**. Tập test được
> chạm vào đúng một lần ở Bước 6. Nếu dò tham số trên test thì điểm số test không còn là ước lượng
> khách quan nữa.

---
## BƯỚC 1 — Môi trường và thư mục kết quả

In [1]:
# Cấu hình số luồng CPU trước khi import NumPy/Pandas.
import os

logical_cpus = os.cpu_count() or 1
compute_threads = max(1, logical_cpus // 2)

os.environ['OPENBLAS_NUM_THREADS'] = str(compute_threads)
os.environ['MKL_NUM_THREADS'] = str(compute_threads)
os.environ['OMP_NUM_THREADS'] = str(compute_threads)
os.environ['NUMEXPR_NUM_THREADS'] = str(compute_threads)

print(f'CPU logic: {logical_cpus} | BLAS: {compute_threads} luồng')

CPU logic: 12 | BLAS: 6 luồng


In [2]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import matplotlib
matplotlib.use('Agg')                     # xuất file ảnh, không cần cửa sổ hiển thị
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.colors import LinearSegmentedColormap

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
)

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

RANDOM_STATE = 42

BASE_DIR    = Path.cwd().parent if Path.cwd().name == 'Decesion_trees_train_model' else Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset_csv'
# Hai thư mục kết quả, tách theo mục đích sử dụng:
EXPORT_DIR = Path.cwd() / 'model_export'      # thứ đem đi triển khai
RESULT_DIR = Path.cwd() / 'training_results'  # thứ dùng để đánh giá, báo cáo
EXPORT_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

print(f'Thư mục dữ liệu       : {DATASET_DIR}')
print(f'Mô hình để triển khai : {EXPORT_DIR}')
print(f'Kết quả huấn luyện    : {RESULT_DIR}')

Thư mục dữ liệu       : D:\01.AI_Security\Camera_AI_ONT\dataset_csv
Mô hình để triển khai : D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\model_export
Kết quả huấn luyện    : D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results


In [3]:
# =========================================================
# 1.2 — Bảng màu và kiểu vẽ dùng chung cho mọi biểu đồ
# =========================================================
SURFACE    = '#fcfcfb'    # nền biểu đồ
INK        = '#0b0b0b'    # chữ chính
INK_2      = '#52514e'    # chữ phụ
MUTED      = '#898781'    # nhãn trục
GRID       = '#e1e0d9'    # lưới (nét mảnh, liền)
BASELINE   = '#c3c2b7'    # trục
SERIES_1   = '#2a78d6'    # xanh dương — chuỗi số liệu 1
SERIES_2   = '#eb6834'    # cam       — chuỗi số liệu 2

# Thang đơn sắc xanh dương, nhạt -> đậm, dùng cho ma trận nhầm lẫn
BLUE_RAMP = ['#cde2fb', '#b7d3f6', '#9ec5f4', '#86b6ef', '#6da7ec',
             '#5598e7', '#3987e5', '#2a78d6', '#256abf', '#1c5cab',
             '#184f95', '#104281', '#0d366b']
CMAP_BLUE = LinearSegmentedColormap.from_list('brand_blue', BLUE_RAMP)

plt.rcParams.update({
    'figure.facecolor':  SURFACE,
    'axes.facecolor':    SURFACE,
    'savefig.facecolor': SURFACE,
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Segoe UI', 'DejaVu Sans', 'Arial'],
    'text.color':        INK,
    'axes.labelcolor':   INK_2,
    'xtick.color':       MUTED,
    'ytick.color':       MUTED,
    'axes.edgecolor':    BASELINE,
    'axes.linewidth':    1.0,
    'grid.color':        GRID,
    'grid.linewidth':    1.0,
    'grid.linestyle':    '-',      # lưới luôn là nét liền, không bao giờ nét đứt
    'figure.dpi':        150,
    'savefig.dpi':       150,
    'savefig.bbox':      'tight',
})

def strip_frame(ax, keep=('left', 'bottom')):
    """Bỏ khung thừa, chỉ giữ lại các trục cần thiết."""
    for side in ('top', 'right', 'bottom', 'left'):
        ax.spines[side].set_visible(side in keep)

def rounded_bar(ax, y, width, height, color, radius_frac=0.35, xmax=1.0):
    """Thanh ngang bo tròn ở đầu mút dữ liệu, vuông ở chân trục."""
    if width <= 0:
        return
    r = min(height * radius_frac, width / 2, xmax * 0.01)
    if width <= 2 * r or r <= 0:
        ax.add_patch(Rectangle((0, y - height / 2), width, height,
                               facecolor=color, edgecolor='none'))
        return
    ax.add_patch(FancyBboxPatch(
        (r, y - height / 2 + r), width - 2 * r, height - 2 * r,
        boxstyle=f'round,pad={r}', facecolor=color, edgecolor='none'))
    # che lại phần bo tròn ở chân trục -> đầu này vuông
    ax.add_patch(Rectangle((0, y - height / 2), r * 1.2, height,
                           facecolor=color, edgecolor='none'))

print('Đã nạp bảng màu và kiểu vẽ.')
print(f'  Nền {SURFACE} | Chuỗi 1 {SERIES_1} | Chuỗi 2 {SERIES_2}')

Đã nạp bảng màu và kiểu vẽ.
  Nền #fcfcfb | Chuỗi 1 #2a78d6 | Chuỗi 2 #eb6834


---
## BƯỚC 2 — Nạp dữ liệu và khôi phục ba tập

Ba tập đã được chia sẵn ở notebook trước và ghi vào cột `split`. Đọc lại theo cột đó để đảm bảo
**đúng những dòng ấy**, không chia lại ngẫu nhiên — nếu chia lại thì mọi kết luận trước đó không
còn đối chiếu được.

In [4]:
# =========================================================
# 2.1 — Nạp file và metadata
# =========================================================
DATA_CSV = DATASET_DIR / 'ids_dataset_ready.csv'
META_CSV = DATASET_DIR / 'preprocessing_meta.json'

assert DATA_CSV.exists(), f'Chưa có {DATA_CSV} — chạy preprocessing_data_balance.ipynb trước.'

meta     = json.loads(META_CSV.read_text(encoding='utf-8'))
FEATURES = meta['features']
LABEL_MAP = meta['label_map']
INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES = [INV_LABEL[0], INV_LABEL[1]]

data = pd.read_csv(DATA_CSV)

print(f'File     : {DATA_CSV.name}  -> {data.shape[0]} dòng x {data.shape[1]} cột')
print(f'Feature  : {len(FEATURES)}')
print(f'Nhãn     : {LABEL_MAP}  (tên hiển thị: {CLASS_NAMES})')
print(f'\nChiến lược cân bằng đã chốt: {meta["balancing"]["strategy"]}')
print(f'Trọng số lớp               : {meta["balancing"]["class_weight"]}')

File     : ids_dataset_ready.csv  -> 4515 dòng x 26 cột
Feature  : 24
Nhãn     : {'normal': 0, 'scanport': 1}  (tên hiển thị: ['normal', 'scanport'])

Chiến lược cân bằng đã chốt: class_weight
Trọng số lớp               : {'0': 0.906483, '1': 1.115032}


In [5]:
# =========================================================
# 2.2 — Tách ba tập theo cột `split`
# =========================================================
def take(split_name):
    part = data[data['split'] == split_name]
    return part[FEATURES].copy(), part['label'].copy()

X_train, y_train = take('train')
X_val,   y_val   = take('val')
X_test,  y_test  = take('test')

overview = pd.DataFrame({
    'số dòng':  [len(y_train), len(y_val), len(y_test)],
    'normal':   [int((y == 0).sum()) for y in (y_train, y_val, y_test)],
    'scanport': [int((y == 1).sum()) for y in (y_train, y_val, y_test)],
}, index=['train', 'val', 'test'])
overview['% scanport'] = (overview['scanport'] / overview['số dòng'] * 100).round(1)

print(overview.to_string())
print(f'\nTổng: {overview["số dòng"].sum()} dòng')

# Trọng số lớp: bù lại chênh lệch số lượng giữa hai lớp
CLASS_WEIGHT = {int(k): v for k, v in meta['balancing']['class_weight'].items()}
print(f'\nTrọng số áp dụng khi train: {CLASS_WEIGHT}')

       số dòng  normal  scanport  % scanport
train     3160    1743      1417        44.8
val        677     374       303        44.8
test       678     374       304        44.8

Tổng: 4515 dòng

Trọng số áp dụng khi train: {0: 0.906483, 1: 1.115032}


### Dữ liệu đã được chuẩn hoá về [0, 1]

Notebook tiền xử lý đã co giãn toàn bộ 24 cột về khoảng [0, 1] bằng `MinMaxScaler`, với min/max
học từ tập train. Điều này kéo theo hai hệ quả mà notebook này phải xử lý:

| Hệ quả | Xử lý ở bước nào |
|---|---|
| Ngưỡng chia của cây giờ là số rất nhỏ (`pkt_len_max <= 0.000518`), đọc không hiểu gì | Bước 9 in thêm bản **quy đổi ngược về đơn vị gốc** |
| Mô hình chỉ chạy đúng nếu dữ liệu mới cũng được chuẩn hoá y hệt | Bước 13 **nhúng luôn phép chuẩn hoá vào file ONNX**, để đầu vào vẫn là giá trị thô |

Ô dưới dựng lại bộ chuẩn hoá từ metadata để hai bước trên dùng.

In [6]:
# =========================================================
# 2.3 — Dựng lại bộ chuẩn hoá từ metadata
# =========================================================
from sklearn.preprocessing import MinMaxScaler

SCALING = meta['scaling']

scaler = MinMaxScaler()
scaler.scale_      = np.array([SCALING['scale_'][f]     for f in FEATURES])
scaler.min_        = np.array([SCALING['min_'][f]       for f in FEATURES])
scaler.data_min_   = np.array([SCALING['data_min'][f]   for f in FEATURES])
scaler.data_range_ = np.array([SCALING['data_range'][f] for f in FEATURES])
scaler.data_max_   = scaler.data_min_ + scaler.data_range_
scaler.n_features_in_    = len(FEATURES)
scaler.feature_names_in_ = np.array(FEATURES, dtype=object)

def to_raw(values, feature):
    """Đưa giá trị đã chuẩn hoá về lại đơn vị gốc."""
    i = FEATURES.index(feature)
    return (np.asarray(values) - scaler.min_[i]) / scaler.scale_[i]

print(f"Phương pháp    : {SCALING['method']} -> {SCALING['feature_range']}")
print(f"Học tham số từ : {SCALING['fitted_on']}")
print(f"Công thức      : {SCALING['formula']}")

# đối chiếu lại: dữ liệu train phải nằm gọn trong [0,1]
print(f'\nKiểm tra tập train: min={X_train.to_numpy().min():.4f}  max={X_train.to_numpy().max():.4f}')
print(f'Kiểm tra tập test : min={X_test.to_numpy().min():.4f}  max={X_test.to_numpy().max():.4f}'
      '   <- vượt 1 là bình thường: giá trị mới lớn hơn khoảng đã thấy lúc train')

Phương pháp    : MinMaxScaler -> [0, 1]
Học tham số từ : train_only
Công thức      : x_scaled = (x - data_min) / data_range  =  x * scale_ + min_

Kiểm tra tập train: min=0.0000  max=1.0000
Kiểm tra tập test : min=0.0000  max=1.2000   <- vượt 1 là bình thường: giá trị mới lớn hơn khoảng đã thấy lúc train


---
## BƯỚC 3 — Mô hình cơ sở

Trước khi tinh chỉnh, cần một mốc để biết việc tinh chỉnh có đáng hay không. Hai mốc:

- **Cây không giới hạn độ sâu** — mọc tự do đến khi mọi lá thuần khiết, gần như chắc chắn học vẹt
- **Cây độ sâu 8** — mức đã dùng xuyên suốt các notebook trước

In [7]:
# =========================================================
# 3.1 — Hai mô hình cơ sở
# =========================================================
def evaluate(model, X, y, tag=''):
    """Trả về các chỉ số chính trên một tập dữ liệu."""
    pred = model.predict(X)
    return {
        'mô hình':   tag,
        'accuracy':  accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall':    recall_score(y, pred, zero_division=0),
        'f1_macro':  f1_score(y, pred, average='macro'),
    }

baselines = {}
rows = []
for tag, depth in [('Cây không giới hạn', None), ('Cây độ sâu 8', 8)]:
    m = DecisionTreeClassifier(max_depth=depth, class_weight=CLASS_WEIGHT,
                               random_state=RANDOM_STATE).fit(X_train, y_train)
    baselines[tag] = m
    r = evaluate(m, X_val, y_val, tag)
    r['độ sâu'] = m.get_depth()
    r['số lá']  = m.get_n_leaves()
    rows.append(r)

base_df = pd.DataFrame(rows)[['mô hình', 'độ sâu', 'số lá', 'accuracy', 'precision', 'recall', 'f1_macro']]
print('Đánh giá trên tập VAL:\n')
print(base_df.round(4).to_string(index=False))

print('\nCây không giới hạn có rất nhiều lá so với cây độ sâu 8 mà điểm không hơn')
print('-> dấu hiệu học vẹt: cây ghi nhớ từng dòng thay vì học quy luật chung.')

Đánh giá trên tập VAL:

           mô hình  độ sâu  số lá  accuracy  precision  recall  f1_macro
Cây không giới hạn      11     40    0.9926     0.9869  0.9967    0.9925
      Cây độ sâu 8       8     34    0.9926     0.9869  0.9967    0.9925

Cây không giới hạn có rất nhiều lá so với cây độ sâu 8 mà điểm không hơn
-> dấu hiệu học vẹt: cây ghi nhớ từng dòng thay vì học quy luật chung.


---
## BƯỚC 4 — Dò siêu tham số trên tập val

Bốn tham số, mỗi tham số kiểm soát một cách chống học vẹt khác nhau:

| Tham số | Ý nghĩa |
|---|---|
| `max_depth` | Chiều sâu tối đa — chặn cây mọc quá sâu |
| `min_samples_leaf` | Số mẫu tối thiểu ở mỗi lá — chặn lá chỉ chứa vài dòng cá biệt |
| `criterion` | Cách đo độ hỗn tạp khi chọn điểm chia (`gini` / `entropy`) |
| `ccp_alpha` | Cắt tỉa sau khi mọc — bỏ nhánh có lợi ích thấp hơn chi phí phức tạp |

Mỗi tổ hợp được **huấn luyện trên train, chấm điểm trên val**. Tập test hoàn toàn không tham gia.

In [8]:
# =========================================================
# 4.1 — Duyệt toàn bộ tổ hợp tham số
# =========================================================
from itertools import product

grid = {
    'max_depth':        [3, 4, 5, 6, 8, 10, 12, None],
    'min_samples_leaf': [1, 2, 5, 10, 20],
    'criterion':        ['gini', 'entropy'],
    'ccp_alpha':        [0.0, 0.0005, 0.001, 0.005],
}
combos = list(product(*grid.values()))
print(f'Số tổ hợp phải thử: {len(combos)}\n')

records = []
for depth, leaf, crit, alpha in combos:
    m = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, criterion=crit,
                               ccp_alpha=alpha, class_weight=CLASS_WEIGHT,
                               random_state=RANDOM_STATE).fit(X_train, y_train)
    records.append({
        'max_depth': depth, 'min_samples_leaf': leaf, 'criterion': crit, 'ccp_alpha': alpha,
        'f1_train':  f1_score(y_train, m.predict(X_train), average='macro'),
        'f1_val':    f1_score(y_val,   m.predict(X_val),   average='macro'),
        'số lá':     m.get_n_leaves(),
    })

search = pd.DataFrame(records)
search['chênh lệch'] = search['f1_train'] - search['f1_val']   # càng lớn càng học vẹt

print('10 tổ hợp tốt nhất theo f1_macro trên val:')
print(search.sort_values('f1_val', ascending=False).head(10).round(4).to_string(index=False))

Số tổ hợp phải thử: 320



10 tổ hợp tốt nhất theo f1_macro trên val:
 max_depth  min_samples_leaf criterion  ccp_alpha  f1_train  f1_val  số lá  chênh lệch
       NaN                 5   entropy     0.0050    0.9946   0.997     10     -0.0025
       NaN                 2   entropy     0.0050    0.9946   0.997     10     -0.0025
       NaN                 1   entropy     0.0050    0.9946   0.997     10     -0.0025
      12.0                 1   entropy     0.0050    0.9946   0.997     10     -0.0025
      12.0                 5   entropy     0.0050    0.9946   0.997     10     -0.0025
       NaN                 1      gini     0.0005    0.9968   0.997     21     -0.0002
      12.0                 2   entropy     0.0050    0.9946   0.997     10     -0.0025
      12.0                 1      gini     0.0005    0.9968   0.997     21     -0.0002
       8.0                 2   entropy     0.0050    0.9946   0.997     10     -0.0025
       8.0                 5   entropy     0.0050    0.9946   0.997     10     -0.0025


In [9]:
# =========================================================
# 4.2 — Chọn tổ hợp thắng cuộc
# =========================================================
# Khi nhiều tổ hợp bằng điểm nhau trên val, ưu tiên cây ĐƠN GIẢN HƠN:
# ít lá hơn -> dễ đọc, dễ giải thích, ít học vẹt hơn.
best_f1 = search['f1_val'].max()
tied = search[np.isclose(search['f1_val'], best_f1)]

print(f'Điểm f1_val cao nhất : {best_f1:.4f}')
print(f'Số tổ hợp bằng điểm  : {len(tied)}')
print('\nTrong số đó, chọn cây ít lá nhất:')

best = tied.sort_values(['số lá', 'chênh lệch']).iloc[0]
BEST_PARAMS = {
    'max_depth':        None if pd.isna(best['max_depth']) else int(best['max_depth']),
    'min_samples_leaf': int(best['min_samples_leaf']),
    'criterion':        best['criterion'],
    'ccp_alpha':        float(best['ccp_alpha']),
}
print(json.dumps(BEST_PARAMS, indent=2, ensure_ascii=False))
print(f"\nSố lá: {int(best['số lá'])} | f1_train {best['f1_train']:.4f} | f1_val {best['f1_val']:.4f} "
      f"| chênh lệch {best['chênh lệch']:.4f}")

Điểm f1_val cao nhất : 0.9970
Số tổ hợp bằng điểm  : 16

Trong số đó, chọn cây ít lá nhất:
{
  "max_depth": 8,
  "min_samples_leaf": 1,
  "criterion": "entropy",
  "ccp_alpha": 0.005
}

Số lá: 10 | f1_train 0.9946 | f1_val 0.9970 | chênh lệch -0.0025


In [10]:
# =========================================================
# 4.3 — Biểu đồ: độ sâu ảnh hưởng thế nào tới học vẹt
# =========================================================
curve = (search[(search['min_samples_leaf'] == BEST_PARAMS['min_samples_leaf']) &
                (search['criterion'] == BEST_PARAMS['criterion']) &
                (search['ccp_alpha'] == BEST_PARAMS['ccp_alpha'])]
         .copy())
curve['depth_num'] = curve['max_depth'].fillna(curve['max_depth'].max() + 4)
curve = curve.sort_values('depth_num')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(curve['depth_num'], curve['f1_train'], color=SERIES_1, lw=2,
        marker='o', ms=8, mfc=SERIES_1, mec=SURFACE, mew=2, label='Tập train', zorder=3)
ax.plot(curve['depth_num'], curve['f1_val'], color=SERIES_2, lw=2,
        marker='o', ms=8, mfc=SERIES_2, mec=SURFACE, mew=2, label='Tập val', zorder=3)

ax.set_xticks(curve['depth_num'])
ax.set_xticklabels([('Không giới hạn' if pd.isna(d) else int(d)) for d in curve['max_depth']], fontsize=9)
ax.set_xlabel('Độ sâu tối đa của cây', fontsize=10)
ax.set_ylabel('f1_macro', fontsize=10)
ax.set_title('Quá 6 tầng thì sâu thêm cũng không hơn — chọn cây đơn giản nhất ở điểm bão hoà',
             fontsize=11.5, color=INK, pad=14, loc='left')
ax.grid(axis='y', lw=1)
ax.set_axisbelow(True)
strip_frame(ax)
ax.legend(frameon=False, fontsize=9, labelcolor=INK_2, loc='lower right')

# chỉ ghi nhãn ở điểm được chọn, không ghi số lên mọi điểm
sel = curve[curve['depth_num'] == (BEST_PARAMS['max_depth'] or curve['depth_num'].max())]
if len(sel):
    ax.annotate(f'đã chọn — {sel.iloc[0]["f1_val"]:.4f}',
                xy=(sel.iloc[0]['depth_num'], sel.iloc[0]['f1_val']),
                xytext=(0, 16), textcoords='offset points',
                ha='center', fontsize=9, color=INK_2)
ax.margins(y=0.10)

print('Lưu ý khi đọc: đường val nằm TRÊN đường train là chuyện bình thường ở đây,')
print('vì ccp_alpha đang cắt tỉa mạnh nên cây không khớp hết được tập train.')
print('Dấu hiệu học vẹt phải nhìn ở chỗ khác: cây sâu hơn mà val không nhích lên.')

fig.savefig(RESULT_DIR / 'tuning_depth_curve.png')
plt.close(fig)
print(f'Đã lưu: {RESULT_DIR / "tuning_depth_curve.png"}')

Lưu ý khi đọc: đường val nằm TRÊN đường train là chuyện bình thường ở đây,
vì ccp_alpha đang cắt tỉa mạnh nên cây không khớp hết được tập train.
Dấu hiệu học vẹt phải nhìn ở chỗ khác: cây sâu hơn mà val không nhích lên.


Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\tuning_depth_curve.png


---
## BƯỚC 5 — Huấn luyện mô hình cuối

Tham số đã chọn xong bằng tập val, nên giờ **gộp train + val** để huấn luyện mô hình cuối — tận
dụng thêm 15% dữ liệu. Tập test vẫn nằm ngoài, chưa bị chạm tới.

In [11]:
# =========================================================
# 5.1 — Huấn luyện trên train + val
# =========================================================
X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)

model = DecisionTreeClassifier(**BEST_PARAMS, class_weight=CLASS_WEIGHT,
                               random_state=RANDOM_STATE).fit(X_full, y_full)

print(f'Dữ liệu huấn luyện : {len(y_full)} dòng (train {len(y_train)} + val {len(y_val)})')
print(f'Tham số            : {BEST_PARAMS}')
print(f'Độ sâu thực tế     : {model.get_depth()}')
print(f'Số lá              : {model.get_n_leaves()}')

# Kiểm tra chéo trên chính tập huấn luyện để có sai số
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_full, y_full, cv=cv, scoring='f1_macro')
print(f'\nKiểm tra chéo 5 phần: f1_macro = {cv_scores.mean():.4f} +- {cv_scores.std():.4f}')

Dữ liệu huấn luyện : 3837 dòng (train 3160 + val 677)
Tham số            : {'max_depth': 8, 'min_samples_leaf': 1, 'criterion': 'entropy', 'ccp_alpha': 0.005}
Độ sâu thực tế     : 6
Số lá              : 9



Kiểm tra chéo 5 phần: f1_macro = 0.9921 +- 0.0024


---
## BƯỚC 6 — Đánh giá trên tập test

Đây là **lần duy nhất** tập test được sử dụng.

In [12]:
# =========================================================
# 6.1 — Báo cáo phân loại
# =========================================================
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

report_txt = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4)
print('KẾT QUẢ TRÊN TẬP TEST\n')
print(report_txt)

TEST_METRICS = {
    'accuracy':          float(accuracy_score(y_test, y_pred)),
    'precision_scan':    float(precision_score(y_test, y_pred, zero_division=0)),
    'recall_scan':       float(recall_score(y_test, y_pred, zero_division=0)),
    'f1_scan':           float(f1_score(y_test, y_pred, zero_division=0)),
    'f1_macro':          float(f1_score(y_test, y_pred, average='macro')),
    'roc_auc':           float(auc(*roc_curve(y_test, y_proba)[:2])),
    'average_precision': float(average_precision_score(y_test, y_proba)),
}
for k, v in TEST_METRICS.items():
    print(f'  {k:18s} {v:.4f}')

# lưu báo cáo dạng văn bản và dạng bảng
(RESULT_DIR / 'classification_report.txt').write_text(
    'BÁO CÁO PHÂN LOẠI — TẬP TEST\n' + '=' * 60 + '\n\n' + report_txt +
    '\n\nTham số mô hình: ' + json.dumps(BEST_PARAMS, ensure_ascii=False),
    encoding='utf-8')

rep_dict = classification_report(y_test, y_pred, target_names=CLASS_NAMES,
                                 digits=4, output_dict=True)
pd.DataFrame(rep_dict).T.round(4).to_csv(RESULT_DIR / 'classification_report.csv',
                                         encoding='utf-8-sig')
print(f'\nĐã lưu: classification_report.txt, classification_report.csv')

KẾT QUẢ TRÊN TẬP TEST

              precision    recall  f1-score   support

      normal     0.9894    1.0000    0.9947       374
    scanport     1.0000    0.9868    0.9934       304

    accuracy                         0.9941       678
   macro avg     0.9947    0.9934    0.9940       678
weighted avg     0.9942    0.9941    0.9941       678

  accuracy           0.9941
  precision_scan     1.0000
  recall_scan        0.9868
  f1_scan            0.9934
  f1_macro           0.9940
  roc_auc            0.9995
  average_precision  0.9990

Đã lưu: classification_report.txt, classification_report.csv


---
## BƯỚC 7 — Ma trận nhầm lẫn

Hai bảng cạnh nhau:

- **Bên trái — số lượng tuyệt đối:** đếm đúng bao nhiêu dòng rơi vào từng ô
- **Bên phải — tỉ lệ theo hàng:** mỗi hàng cộng lại bằng 100%, cho biết *trong số các flow thực sự
  thuộc lớp này, mô hình đoán đúng bao nhiêu phần trăm* (chính là recall của từng lớp)

Ô quan trọng nhất với một hệ phát hiện xâm nhập là **góc dưới bên trái**: flow tấn công thật nhưng
bị bỏ sót — nguy hiểm hơn nhiều so với báo động nhầm.

In [13]:
# =========================================================
# 7.1 — Tính ma trận
# =========================================================
cm      = confusion_matrix(y_test, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

cm_df = pd.DataFrame(cm,
                     index=[f'thực tế: {c}' for c in CLASS_NAMES],
                     columns=[f'dự đoán: {c}' for c in CLASS_NAMES])
print('Số lượng tuyệt đối:')
print(cm_df.to_string())

tn, fp, fn, tp = cm.ravel()
print(f'\n  Đúng normal   (TN): {tn:>4}')
print(f'  Báo nhầm      (FP): {fp:>4}  <- flow bình thường bị coi là tấn công')
print(f'  BỎ SÓT        (FN): {fn:>4}  <- flow tấn công không bị phát hiện')
print(f'  Bắt đúng      (TP): {tp:>4}')
print(f'\n  Tỉ lệ bỏ sót  : {fn/(fn+tp):.2%} số flow tấn công')
print(f'  Tỉ lệ báo nhầm: {fp/(fp+tn):.2%} số flow bình thường')

cm_df.to_csv(RESULT_DIR / 'confusion_matrix.csv', encoding='utf-8-sig')
pd.DataFrame(cm_norm.round(4),
             index=cm_df.index, columns=cm_df.columns
             ).to_csv(RESULT_DIR / 'confusion_matrix_normalized.csv', encoding='utf-8-sig')
print(f'\nĐã lưu: confusion_matrix.csv, confusion_matrix_normalized.csv')

Số lượng tuyệt đối:
                   dự đoán: normal  dự đoán: scanport
thực tế: normal                374                  0
thực tế: scanport                4                300

  Đúng normal   (TN):  374
  Báo nhầm      (FP):    0  <- flow bình thường bị coi là tấn công
  BỎ SÓT        (FN):    4  <- flow tấn công không bị phát hiện
  Bắt đúng      (TP):  300

  Tỉ lệ bỏ sót  : 1.32% số flow tấn công
  Tỉ lệ báo nhầm: 0.00% số flow bình thường

Đã lưu: confusion_matrix.csv, confusion_matrix_normalized.csv


In [14]:
# =========================================================
# 7.2 — Vẽ ma trận nhầm lẫn
# =========================================================
def draw_matrix(ax, values, annots, title, vmax):
    """Vẽ lưới ô màu; khoảng hở giữa các ô để trắng chứ không kẻ viền."""
    n = values.shape[0]
    gap = 0.004                                   # khoảng hở màu nền ~2px giữa 2 ô
    for i in range(n):
        for j in range(n):
            shade = values[i, j] / vmax if vmax else 0
            face  = CMAP_BLUE(0.12 + 0.80 * shade)
            ax.add_patch(Rectangle((j + gap, n - 1 - i + gap),
                                   1 - 2 * gap, 1 - 2 * gap,
                                   facecolor=face, edgecolor='none'))
            # chữ trắng trên nền đậm, chữ đen trên nền nhạt
            ax.text(j + 0.5, n - 1 - i + 0.5, annots[i][j],
                    ha='center', va='center', fontsize=13,
                    color='#ffffff' if shade > 0.55 else INK,
                    linespacing=1.5)
    ax.set_xlim(0, n); ax.set_ylim(0, n)
    ax.set_xticks([k + 0.5 for k in range(n)])
    ax.set_yticks([n - 1 - k + 0.5 for k in range(n)])
    ax.set_xticklabels([f'dự đoán\n{c}' for c in CLASS_NAMES], fontsize=10)
    ax.set_yticklabels([f'thực tế\n{c}' for c in CLASS_NAMES], fontsize=10)
    ax.set_title(title, fontsize=12, color=INK, pad=14, loc='left')
    ax.tick_params(length=0)
    strip_frame(ax, keep=())
    ax.set_aspect('equal')

annot_cnt  = [[f'{cm[i][j]:,}' for j in range(2)] for i in range(2)]
annot_pct  = [[f'{cm_norm[i][j]:.1%}' for j in range(2)] for i in range(2)]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
draw_matrix(axes[0], cm,      annot_cnt, 'Số lượng flow',        cm.max())
draw_matrix(axes[1], cm_norm, annot_pct, 'Tỉ lệ theo hàng (%)',  1.0)
fig.suptitle('Ma trận nhầm lẫn — tập test', fontsize=14, color=INK, x=0.065, ha='left', y=1.02)
fig.savefig(RESULT_DIR / 'confusion_matrix.png')
plt.close(fig)

print(f'Đã lưu: {RESULT_DIR / "confusion_matrix.png"}')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\confusion_matrix.png


---
## BƯỚC 8 — % ảnh hưởng của từng feature

Đo bằng **hai cách khác nhau**, vì mỗi cách có điểm mù riêng:

| Cách đo | Nguyên lý | Điểm yếu |
|---|---|---|
| **Theo độ giảm hỗn tạp** (`feature_importances_`) | Cộng dồn mức độ "làm sạch" dữ liệu mà mỗi feature đạt được tại các điểm chia | Thiên vị feature có nhiều giá trị phân biệt; tính trên chính dữ liệu train |
| **Theo hoán vị** (`permutation_importance`) | Xáo trộn ngẫu nhiên một cột rồi đo điểm số tụt bao nhiêu | Chậm hơn; nếu hai feature trùng thông tin thì cả hai cùng bị đánh giá thấp |

Cách thứ nhất cộng lại đúng 100% nên trả lời trực tiếp câu hỏi "phần trăm ảnh hưởng". Cách thứ hai
đo **trên tập test**, nên nói lên khả năng khái quát hoá thật sự. Hai cách lệch nhau nhiều là dấu
hiệu cần xem lại.

In [15]:
# =========================================================
# 8.1 — Tính độ quan trọng theo cả hai cách
# =========================================================
imp_gini = pd.Series(model.feature_importances_, index=FEATURES)
imp_pct  = imp_gini * 100

perm = permutation_importance(model, X_test, y_test, n_repeats=30,
                              random_state=RANDOM_STATE, scoring='f1_macro')

fi = pd.DataFrame({
    'feature':            FEATURES,
    '% ảnh hưởng':        imp_pct.values,
    'hoán vị (trung bình)': perm.importances_mean,
    'hoán vị (độ lệch)':  perm.importances_std,
}).sort_values('% ảnh hưởng', ascending=False).reset_index(drop=True)

fi['% cộng dồn'] = fi['% ảnh hưởng'].cumsum()
fi.insert(0, 'hạng', range(1, len(fi) + 1))

print('ẢNH HƯỞNG CỦA TỪNG FEATURE (tổng = 100%)\n')
print(fi.round(4).to_string(index=False))

fi.round(6).to_csv(RESULT_DIR / 'feature_importance.csv', index=False, encoding='utf-8-sig')
print(f'\nĐã lưu: {RESULT_DIR / "feature_importance.csv"}')

n_used = int((fi['% ảnh hưởng'] > 0).sum())
print(f'\nSố feature cây thực sự dùng : {n_used} / {len(FEATURES)}')
print(f'Số feature không được dùng  : {len(FEATURES) - n_used}')
top3 = fi.head(3)['% ảnh hưởng'].sum()
print(f'3 feature đầu chiếm         : {top3:.1f}% tổng ảnh hưởng')

ẢNH HƯỞNG CỦA TỪNG FEATURE (tổng = 100%)

 hạng               feature  % ảnh hưởng  hoán vị (trung bình)  hoán vị (độ lệch)  % cộng dồn
    1           pkt_len_max      78.3637                0.4959             0.0164     78.3637
    2          bwd_iat_mean       7.2424                0.0245             0.0028     85.6061
    3          pkt_len_mean       6.8825                0.0105             0.0017     92.4886
    4      bwd_pkt_len_mean       6.0448                0.0300             0.0023     98.5334
    5          flow_iat_min       0.7445                0.0038             0.0019     99.2779
    6    down_up_byte_ratio       0.7221                0.0053             0.0029    100.0000
    7      flow_bytes_per_s       0.0000                0.0000             0.0000    100.0000
    8       flow_pkts_per_s       0.0000                0.0000             0.0000    100.0000
    9          fwd_iat_mean       0.0000                0.0000             0.0000    100.0000
   10           fw

In [16]:
# =========================================================
# 8.2 — Biểu đồ % ảnh hưởng
# =========================================================
# Feature là các hạng mục KHÔNG có thứ tự tự nhiên -> tất cả thanh dùng CHUNG một màu.
# Tô đậm nhạt theo giá trị sẽ mã hoá lặp lại đúng thông tin mà chiều dài thanh đã thể hiện.
plot_df = fi.sort_values('% ảnh hưởng')          # nhỏ ở dưới, lớn ở trên
ys = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(9, 8))
xmax = plot_df['% ảnh hưởng'].max()

for y, (val, name) in enumerate(zip(plot_df['% ảnh hưởng'], plot_df['feature'])):
    if val > 0:
        rounded_bar(ax, y, val, 0.42, SERIES_1, xmax=xmax)

# giá trị đặt ở đầu mút thanh — đúng quy ước cho biểu đồ thanh ngang
for y, val in enumerate(plot_df['% ảnh hưởng']):
    if val > 0.01:
        ax.text(val + xmax * 0.015, y, f'{val:.1f}%', va='center', ha='left',
                fontsize=9, color=INK_2)
    else:
        ax.text(xmax * 0.015, y, 'không dùng', va='center', ha='left',
                fontsize=8, color=MUTED, style='italic')

ax.set_yticks(ys)
ax.set_yticklabels(plot_df['feature'], fontsize=9.5, color=INK_2)
ax.set_xlim(0, xmax * 1.20)
ax.set_ylim(-0.8, len(plot_df) - 0.2)
ax.set_xlabel('Phần trăm ảnh hưởng tới quyết định của cây (%)', fontsize=10)
ax.set_title('Feature nào quyết định kết quả phân loại',
             fontsize=13, color=INK, pad=16, loc='left')
ax.grid(axis='x', lw=1)
ax.set_axisbelow(True)
ax.tick_params(length=0)
strip_frame(ax, keep=('left',))

fig.savefig(RESULT_DIR / 'feature_importance.png')
plt.close(fig)
print(f'Đã lưu: {RESULT_DIR / "feature_importance.png"}')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\feature_importance.png


In [17]:
# =========================================================
# 8.3 — Đối chiếu hai cách đo
# =========================================================
comp = fi[fi['% ảnh hưởng'] > 0].copy()
ys = np.arange(len(comp))[::-1]

# chiều cao hình tính theo số dòng để bề dày thanh giữ ở mức ~22px, không phình to
fig, axes = plt.subplots(1, 2, figsize=(12, 0.33 * len(comp) + 2.0), sharey=True)

# trái: theo độ giảm hỗn tạp
xm1 = comp['% ảnh hưởng'].max()
for y, val in zip(ys, comp['% ảnh hưởng']):
    rounded_bar(axes[0], y, val, 0.42, SERIES_1, xmax=xm1)
axes[0].set_xlim(0, xm1 * 1.18)
axes[0].set_xlabel('% ảnh hưởng (độ giảm hỗn tạp)', fontsize=10)
axes[0].set_title('Đo trên tập huấn luyện', fontsize=11, color=INK, pad=12, loc='left')

# phải: theo hoán vị, có thanh sai số
xm2 = max(comp['hoán vị (trung bình)'].max(), 1e-9)
for y, val in zip(ys, comp['hoán vị (trung bình)']):
    rounded_bar(axes[1], y, max(val, 0), 0.42, SERIES_2, xmax=xm2)
axes[1].errorbar(comp['hoán vị (trung bình)'], ys,
                 xerr=comp['hoán vị (độ lệch)'], fmt='none',
                 ecolor=INK_2, elinewidth=1.2, capsize=3, alpha=0.75)
axes[1].set_xlim(0, xm2 * 1.25)
axes[1].set_xlabel('Mức tụt f1_macro khi xáo trộn cột', fontsize=10)
axes[1].set_title('Đo trên tập test', fontsize=11, color=INK, pad=12, loc='left')

for ax in axes:
    ax.grid(axis='x', lw=1); ax.set_axisbelow(True); ax.tick_params(length=0)
    strip_frame(ax, keep=('left',))
axes[0].set_yticks(ys)
axes[0].set_yticklabels(comp['feature'], fontsize=9.5, color=INK_2)
axes[0].set_ylim(min(ys) - 0.8, max(ys) + 0.8)

fig.suptitle('Hai cách đo độ quan trọng — khớp nhau thì kết luận mới đáng tin',
             fontsize=13, color=INK, x=0.045, ha='left', y=1.0)
fig.savefig(RESULT_DIR / 'feature_importance_comparison.png')
plt.close(fig)
print(f'Đã lưu: {RESULT_DIR / "feature_importance_comparison.png"}')

rank_gini = comp.set_index('feature')['% ảnh hưởng'].rank(ascending=False)
rank_perm = comp.set_index('feature')['hoán vị (trung bình)'].rank(ascending=False)
print(f'\nTương quan thứ hạng giữa hai cách đo (Spearman): {rank_gini.corr(rank_perm, method="spearman"):.3f}')
print('  gần 1.0 = hai cách đo đồng thuận')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\feature_importance_comparison.png

Tương quan thứ hạng giữa hai cách đo (Spearman): 0.771
  gần 1.0 = hai cách đo đồng thuận


---
## BƯỚC 9 — Kiểm tra rò rỉ

Đúng Bước 4 và 5 trong quy trình của file Excel: đọc luật của cây và soi feature quan trọng nhất
bằng con mắt nghi ngờ.

Câu hỏi cần trả lời: cây đang chia theo **hành vi** (nhịp gói, cờ TCP, tốc độ) hay theo **dấu vết
định danh** (thứ gắn với đúng thiết bị/phiên capture này)?

In [18]:
# =========================================================
# 9.1 — In ra luật của cây
# =========================================================
import copy

# Cây học trên dữ liệu đã chuẩn hoá nên mọi ngưỡng đều là số rất nhỏ, đọc không ra ý nghĩa.
# Tạo một bản sao rồi quy đổi từng ngưỡng về đơn vị gốc: x_raw = (x_scaled - min_) / scale_
model_raw_units = copy.deepcopy(model)
_t = model_raw_units.tree_
for _i in range(_t.node_count):
    _f = _t.feature[_i]
    if _f >= 0:                                   # bỏ qua nút lá
        _t.threshold[_i] = (_t.threshold[_i] - scaler.min_[_f]) / scaler.scale_[_f]

rules_scaled = export_text(model, feature_names=list(FEATURES), decimals=6)
rules_raw    = export_text(model_raw_units, feature_names=list(FEATURES), decimals=3)

print('LUẬT CỦA CÂY — theo giá trị ĐÃ CHUẨN HOÁ (đúng thứ mô hình thấy)\n')
print(export_text(model, feature_names=list(FEATURES), max_depth=2, decimals=6))

print('\nCÙNG CÂY ĐÓ — quy đổi về ĐƠN VỊ GỐC (đọc được bằng mắt thường)\n')
print(export_text(model_raw_units, feature_names=list(FEATURES), max_depth=2, decimals=2))

(RESULT_DIR / 'decision_tree_rules.txt').write_text(
    'LUẬT ĐẦY ĐỦ CỦA CÂY QUYẾT ĐỊNH\n' + '=' * 66 +
    f'\nTham số : {json.dumps(BEST_PARAMS, ensure_ascii=False)}' +
    f'\nĐộ sâu  : {model.get_depth()} | Số lá: {model.get_n_leaves()}\n\n' +
    'PHẦN 1 — NGƯỠNG THEO ĐƠN VỊ GỐC (byte, giây, số đếm)\n' +
    'Đây là bản để đọc và kiểm chứng bằng kiến thức mạng.\n' + '-' * 66 + '\n' +
    rules_raw +
    '\n\nPHẦN 2 — NGƯỠNG THEO GIÁ TRỊ ĐÃ CHUẨN HOÁ [0,1]\n' +
    'Đây là bản đúng với những gì mô hình thực sự tính toán.\n' + '-' * 66 + '\n' +
    rules_scaled,
    encoding='utf-8')
print(f'\nĐã lưu cả hai bản: {RESULT_DIR / "decision_tree_rules.txt"}')

LUẬT CỦA CÂY — theo giá trị ĐÃ CHUẨN HOÁ (đúng thứ mô hình thấy)

|--- pkt_len_max <= 0.000518
|   |--- pkt_len_mean <= 0.006379
|   |   |--- class: 0
|   |--- pkt_len_mean >  0.006379
|   |   |--- class: 1
|--- pkt_len_max >  0.000518
|   |--- bwd_iat_mean <= 0.000002
|   |   |--- class: 0
|   |--- bwd_iat_mean >  0.000002
|   |   |--- pkt_len_max <= 0.018447
|   |   |   |--- truncated branch of depth 4
|   |   |--- pkt_len_max >  0.018447
|   |   |   |--- class: 0


CÙNG CÂY ĐÓ — quy đổi về ĐƠN VỊ GỐC (đọc được bằng mắt thường)

|--- pkt_len_max <= 60.00
|   |--- pkt_len_mean <= 55.67
|   |   |--- class: 0
|   |--- pkt_len_mean >  55.67
|   |   |--- class: 1
|--- pkt_len_max >  60.00
|   |--- bwd_iat_mean <= 0.00
|   |   |--- class: 0
|   |--- bwd_iat_mean >  0.00
|   |   |--- pkt_len_max <= 683.50
|   |   |   |--- truncated branch of depth 4
|   |   |--- pkt_len_max >  683.50
|   |   |   |--- class: 0


Đã lưu cả hai bản: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\tr

In [19]:
# =========================================================
# 9.2 — Soi feature đứng đầu
# =========================================================
top_feat, top_val = fi.iloc[0]['feature'], fi.iloc[0]['% ảnh hưởng']
print(f'Feature ảnh hưởng lớn nhất: {top_feat} ({top_val:.1f}%)\n')

RISKY = {'fwd_init_win', 'bwd_init_win', 'fwd_min_seg_size', 'dst_port', 'src_port'}
flags = []

if top_val > 50:
    flags.append(f'"{top_feat}" một mình chiếm {top_val:.1f}% — vượt ngưỡng cảnh báo 50% của file Excel.')
found_risky = RISKY & set(fi[fi['% ảnh hưởng'] > 0]['feature'])
if found_risky:
    flags.append(f'Có feature thuộc nhóm nguy cơ vân tay thiết bị: {found_risky}')

print('Kết quả kiểm tra:')
if flags:
    for f in flags:
        print(f'  [!] {f}')
else:
    print('  Không phát hiện dấu hiệu rò rỉ rõ ràng.')

# Thử bỏ hẳn feature đứng đầu: nếu điểm sập hoàn toàn thì mô hình chỉ dựa vào đúng một thứ
without = [f for f in FEATURES if f != top_feat]
m_wo = DecisionTreeClassifier(**BEST_PARAMS, class_weight=CLASS_WEIGHT,
                              random_state=RANDOM_STATE).fit(X_full[without], y_full)
f1_wo = f1_score(y_test, m_wo.predict(X_test[without]), average='macro')

print(f'\nPhép thử bỏ hẳn "{top_feat}":')
print(f'  f1_macro đầy đủ {len(FEATURES)} feature : {TEST_METRICS["f1_macro"]:.4f}')
print(f'  f1_macro khi bỏ feature đứng đầu     : {f1_wo:.4f}')
print(f'  Mức sụt giảm                          : {TEST_METRICS["f1_macro"] - f1_wo:.4f}')
if f1_wo > 0.95:
    print('  -> Vẫn rất cao. Tín hiệu phân tán trên nhiều feature, không phụ thuộc một cột duy nhất.')
else:
    print('  -> Sụt mạnh. Mô hình phụ thuộc nặng vào một feature duy nhất, cần xem xét lại.')

Feature ảnh hưởng lớn nhất: pkt_len_max (78.4%)

Kết quả kiểm tra:
  [!] "pkt_len_max" một mình chiếm 78.4% — vượt ngưỡng cảnh báo 50% của file Excel.

Phép thử bỏ hẳn "pkt_len_max":
  f1_macro đầy đủ 24 feature : 0.9940
  f1_macro khi bỏ feature đứng đầu     : 0.9925
  Mức sụt giảm                          : 0.0015
  -> Vẫn rất cao. Tín hiệu phân tán trên nhiều feature, không phụ thuộc một cột duy nhất.


In [20]:
# =========================================================
# 9.3 — Vẽ sơ đồ cây (3 tầng đầu cho dễ đọc)
# =========================================================
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(model, feature_names=list(FEATURES), class_names=CLASS_NAMES,
          max_depth=3, filled=True, rounded=True, fontsize=9,
          impurity=False, proportion=True, ax=ax)
ax.set_title('Cây quyết định — 3 tầng đầu', fontsize=14, color=INK, loc='left', pad=16)
fig.savefig(RESULT_DIR / 'decision_tree.png')
plt.close(fig)
print(f'Đã lưu: {RESULT_DIR / "decision_tree.png"}')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\decision_tree.png


---
## BƯỚC 10 — Đường ROC và Precision–Recall

Hai đường này cho biết mô hình hoạt động ra sao **ở mọi ngưỡng quyết định**, chứ không chỉ ở ngưỡng
mặc định 0,5. Với hệ phát hiện xâm nhập, đây là căn cứ để chỉnh ngưỡng theo hướng chấp nhận báo nhầm
nhiều hơn để bỏ sót ít đi.

In [21]:
# =========================================================
# 10.1 — Vẽ hai đường
# =========================================================
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
baseline_rate = (y_test == 1).mean()

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5))

axes[0].plot([0, 1], [0, 1], color=BASELINE, lw=1)          # đoán ngẫu nhiên
axes[0].plot(fpr, tpr, color=SERIES_1, lw=2)
axes[0].fill_between(fpr, tpr, alpha=0.10, color=SERIES_1, lw=0)
axes[0].set_xlabel('Tỉ lệ báo nhầm (FPR)', fontsize=10)
axes[0].set_ylabel('Tỉ lệ bắt đúng (TPR)', fontsize=10)
axes[0].set_title(f'Đường ROC — diện tích dưới đường {roc_auc_val:.4f}',
                  fontsize=11, color=INK, pad=12, loc='left')
axes[0].text(0.62, 0.18, 'đoán ngẫu nhiên', fontsize=8.5, color=MUTED, rotation=32)

axes[1].axhline(baseline_rate, color=BASELINE, lw=1)
axes[1].plot(rec, prec, color=SERIES_1, lw=2)
axes[1].fill_between(rec, prec, alpha=0.10, color=SERIES_1, lw=0)
axes[1].set_xlabel('Recall — bắt được bao nhiêu flow tấn công', fontsize=10)
axes[1].set_ylabel('Precision — báo động có bao nhiêu là thật', fontsize=10)
axes[1].set_title(f'Đường Precision–Recall — AP {ap:.4f}',
                  fontsize=11, color=INK, pad=12, loc='left')
axes[1].text(0.05, baseline_rate + 0.02, f'mức nền {baseline_rate:.1%}',
             fontsize=8.5, color=MUTED)

for ax in axes:
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.grid(lw=1); ax.set_axisbelow(True)
    strip_frame(ax)

fig.suptitle('Chất lượng phân loại ở mọi ngưỡng quyết định',
             fontsize=13, color=INK, x=0.045, ha='left', y=1.0)
fig.savefig(RESULT_DIR / 'roc_pr_curves.png')
plt.close(fig)
print(f'Đã lưu: {RESULT_DIR / "roc_pr_curves.png"}')
print(f'  ROC AUC          : {roc_auc_val:.4f}')
print(f'  Average Precision: {ap:.4f}')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\roc_pr_curves.png
  ROC AUC          : 0.9995
  Average Precision: 0.9990


---
## BƯỚC 11 — Đối chiếu với Random Forest

Decision Tree đơn có ưu điểm lớn là **đọc được luật của nó** — quan trọng với bài toán an ninh, vì
cần giải thích được vì sao một flow bị gắn cờ. Nhưng cần biết cái giá phải trả: một mô hình mạnh hơn
sẽ hơn bao nhiêu?

Nếu khoảng cách nhỏ thì chọn Decision Tree là hợp lý.

In [22]:
# =========================================================
# 11.1 — So sánh ba mô hình trên cùng tập test
# =========================================================
rf = RandomForestClassifier(n_estimators=300, max_depth=BEST_PARAMS['max_depth'],
                            min_samples_leaf=BEST_PARAMS['min_samples_leaf'],
                            class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE,
                            n_jobs=-1).fit(X_full, y_full)

rows = [
    evaluate(baselines['Cây độ sâu 8'], X_test, y_test, 'Cây cơ sở (độ sâu 8)'),
    evaluate(model, X_test, y_test, 'Cây đã tinh chỉnh'),
    evaluate(rf, X_test, y_test, 'Random Forest (300 cây)'),
]
cmp_df = pd.DataFrame(rows)
cmp_df['số lá'] = [baselines['Cây độ sâu 8'].get_n_leaves(), model.get_n_leaves(), np.nan]
cmp_df['đọc được luật'] = ['có', 'có', 'không']

print('So sánh trên TẬP TEST:\n')
print(cmp_df.round(4).to_string(index=False))

gap = cmp_df.iloc[2]['f1_macro'] - cmp_df.iloc[1]['f1_macro']
print(f'\nRandom Forest hơn cây đã tinh chỉnh: {gap:+.4f} f1_macro')
if abs(gap) < 0.01:
    print('-> Khoảng cách không đáng kể. Chọn Decision Tree là hợp lý:')
    print('   đổi lấy khả năng giải thích được từng quyết định mà gần như không mất độ chính xác.')

cmp_df.round(4).to_csv(RESULT_DIR / 'model_comparison.csv', index=False, encoding='utf-8-sig')
print(f'\nĐã lưu: {RESULT_DIR / "model_comparison.csv"}')

So sánh trên TẬP TEST:

                mô hình  accuracy  precision  recall  f1_macro  số lá đọc được luật
   Cây cơ sở (độ sâu 8)    0.9897     0.9837  0.9934    0.9896   34.0            có
      Cây đã tinh chỉnh    0.9941     1.0000  0.9868    0.9940    9.0            có
Random Forest (300 cây)    0.9912     0.9934  0.9868    0.9910    NaN         không

Random Forest hơn cây đã tinh chỉnh: -0.0030 f1_macro
-> Khoảng cách không đáng kể. Chọn Decision Tree là hợp lý:
   đổi lấy khả năng giải thích được từng quyết định mà gần như không mất độ chính xác.

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\model_comparison.csv


---
## BƯỚC 12 — Lưu mô hình và tổng hợp

In [23]:
# =========================================================
# 12.1 — Lưu mô hình kèm mọi thứ cần cho lúc suy luận
# =========================================================
bundle = {
    'model':        model,
    'features':     FEATURES,          # đúng thứ tự cột — sai thứ tự là sai kết quả
    'label_map':    LABEL_MAP,
    'class_names':  CLASS_NAMES,
    'params':       BEST_PARAMS,
    'class_weight': CLASS_WEIGHT,
    'impute_values': meta['imputation']['values'],   # để làm sạch dữ liệu mới y hệt lúc train
}
joblib.dump(bundle, EXPORT_DIR / 'decision_tree_model.joblib')
print(f'Đã lưu mô hình: {EXPORT_DIR / "decision_tree_model.joblib"}')

summary = {
    'model_type': 'DecisionTreeClassifier',
    'params': BEST_PARAMS,
    'class_weight': CLASS_WEIGHT,
    'n_features': len(FEATURES),
    'features': FEATURES,
    'data': {
        'train': int(len(y_train)), 'val': int(len(y_val)), 'test': int(len(y_test)),
        'trained_on': 'train + val', 'n_trained': int(len(y_full)),
    },
    'tree': {'depth': int(model.get_depth()), 'n_leaves': int(model.get_n_leaves()),
             'n_features_used': n_used},
    'test_metrics': TEST_METRICS,
    'cross_val': {'f1_macro_mean': float(cv_scores.mean()), 'f1_macro_std': float(cv_scores.std())},
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'feature_importance_pct': {r['feature']: round(r['% ảnh hưởng'], 4)
                               for _, r in fi.iterrows()},
    'leakage_check': {
        'top_feature': top_feat,
        'top_feature_pct': round(float(top_val), 4),
        'f1_without_top_feature': round(float(f1_wo), 4),
        'flags': flags,
    },
    'comparison_vs_random_forest': {'f1_gap': round(float(gap), 4)},
    'caveat': ('Dữ liệu chỉ có MỘT phiên nmap duy nhất. Điểm số cao phản ánh khả năng nhận diện '
               'đúng phiên quét đó, chưa chứng minh được khả năng khái quát hoá sang các kiểu quét khác.'),
}

(RESULT_DIR / 'model_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Đã lưu tổng hợp: {RESULT_DIR / "model_summary.json"}')

Đã lưu mô hình: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\model_export\decision_tree_model.joblib
Đã lưu tổng hợp: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\training_results\model_summary.json


---
## BƯỚC 13 — Xuất mô hình sang ONNX

ONNX cho phép chạy mô hình ở nơi không có Python/scikit-learn — C++, C#, Java, hoặc ngay trên
thiết bị nhúng. Hợp đồng vào/ra được đặt đúng như yêu cầu:

| | Tên | Kiểu | Kích thước |
|---|---|---|---|
| Đầu vào | `input` | `float32` | `[N, 24]` — **giá trị thô, chưa chuẩn hoá** |
| Đầu ra | `output` | `int64` | `[N]` — 0 = normal, 1 = scanport |

### Phép chuẩn hoá được nhúng vào trong file ONNX

Mô hình được huấn luyện trên dữ liệu đã co giãn về [0, 1]. Nếu file ONNX chỉ chứa mỗi cái cây thì
bên triển khai sẽ phải tự cài lại phép chuẩn hoá với đúng 48 hằng số `scale_` và `min_` — sai một
con số là mô hình cho kết quả sai mà **không hề báo lỗi**.

Cách tránh: gói `MinMaxScaler` và cây vào chung một `Pipeline` rồi mới chuyển sang ONNX. Đồ thị
ONNX khi đó có thêm hai toán tử `Mul` và `Add` thực hiện phép chuẩn hoá ngay bên trong, nên **đầu
vào là giá trị thô** — bên triển khai không cần biết gì về chuẩn hoá nữa.

**Vẫn còn hai điều bắt buộc phải đúng:**

1. **Thứ tự cột.** ONNX chỉ nhận một mảng số, không có tên cột. Đưa sai thứ tự thì mô hình vẫn
   chạy và vẫn trả về kết quả — chỉ là kết quả sai mà không báo lỗi gì. Thứ tự chuẩn nằm trong
   `onnx_io_spec.json`.
2. **Điền giá trị thiếu.** Phép chuẩn hoá đã nằm trong mô hình, nhưng việc làm sạch (tốc độ không
   xác định, IAT âm) và điền chỗ trống bằng trung vị thì **chưa** — vẫn phải làm trước khi gọi.
   Các giá trị trung vị nằm trong `onnx_io_spec.json` và `scaler_params.csv`.

In [24]:
# =========================================================
# 13.1 — Chuyển đổi sang ONNX
# =========================================================
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType, Int64TensorType
    import onnx
    import onnxruntime as ort
except ImportError:
    raise SystemExit('Thiếu thư viện. Chạy: pip install skl2onnx onnxruntime')

# True  -> thêm đầu ra thứ hai `probabilities` [N,2], dùng khi cần tự chỉnh ngưỡng cảnh báo
# False -> đúng một đầu vào `input` và một đầu ra `output`
KEEP_PROBABILITIES = False

ONNX_PATH = EXPORT_DIR / 'decision_tree_model.onnx'

from sklearn.pipeline import Pipeline

# Gói bộ chuẩn hoá và cây vào chung một pipeline -> ONNX nhận GIÁ TRỊ THÔ
export_pipeline = Pipeline([('scaler', scaler), ('clf', model)])

onnx_model = convert_sklearn(
    export_pipeline,
    initial_types=[('input', FloatTensorType([None, len(FEATURES)]))],
    final_types=[('output',        Int64TensorType([None])),
                 ('probabilities', FloatTensorType([None, len(CLASS_NAMES)]))],
    options={id(model): {'zipmap': False}},   # trả về mảng thẳng, không bọc thành danh sách dict
    target_opset=17,
)

if not KEEP_PROBABILITIES:
    del onnx_model.graph.output[1]            # chỉ giữ lại `output`

onnx.checker.check_model(onnx_model)
onnx.save(onnx_model, ONNX_PATH)

print(f'Đã lưu: {ONNX_PATH}  ({ONNX_PATH.stat().st_size / 1024:.1f} KB)')

# Mô hình cây dùng toán tử TreeEnsembleClassifier thuộc miền ai.onnx.ml.
# Phiên bản của miền đó mới là thứ quyết định runtime có chạy được hay không.
OPSETS = {(o.domain or 'ai.onnx'): o.version for o in onnx_model.opset_import}
print(f'Opset  : ' + ', '.join(f'{d} v{v}' for d, v in OPSETS.items()))
print(f'Toán tử: {[n.op_type for n in onnx_model.graph.node]}\n')

def shape_of(v):
    return [(d.dim_value if d.dim_value else 'N') for d in v.type.tensor_type.shape.dim]

print('Đầu vào:')
for v in onnx_model.graph.input:
    print(f'  {v.name:16s} float32  {shape_of(v)}')
print('Đầu ra:')
for v in onnx_model.graph.output:
    print(f'  {v.name:16s}          {shape_of(v)}')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\model_export\decision_tree_model.onnx  (1.6 KB)
Opset  : ai.onnx v17, ai.onnx.ml v1
Toán tử: ['Cast', 'Mul', 'Add', 'TreeEnsembleClassifier', 'Cast', 'Cast']

Đầu vào:
  input            float32  ['N', 24]
Đầu ra:
  output                    ['N']


In [25]:
# =========================================================
# 13.2 — Đối chiếu ONNX với scikit-learn trên toàn bộ tập test
# =========================================================
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])

# X_test đang ở dạng ĐÃ chuẩn hoá -> đưa ngược về giá trị thô,
# vì file ONNX tự chuẩn hoá bên trong.
X_test_raw = (X_test[FEATURES].to_numpy() - scaler.min_) / scaler.scale_
X_onnx = X_test_raw.astype(np.float32)      # thứ tự cột đúng như FEATURES, kiểu float32

print(f'Đầu vào đưa cho ONNX (giá trị thô): min={X_onnx.min():,.2f}  max={X_onnx.max():,.2f}')
print(f'So với dữ liệu đã chuẩn hoá       : min={X_test.to_numpy().min():.4f}  '
      f'max={X_test.to_numpy().max():.4f}\n')

onnx_out = sess.run(None, {'input': X_onnx})
onnx_pred = np.asarray(onnx_out[0]).ravel()

sk_pred = model.predict(X_test)
n_diff = int((onnx_pred != sk_pred).sum())

print(f'Số mẫu kiểm tra      : {len(sk_pred)}')
print(f'Số dự đoán lệch nhau : {n_diff}')
print(f'Accuracy của ONNX    : {accuracy_score(y_test, onnx_pred):.4f}')
print(f'Accuracy của sklearn : {accuracy_score(y_test, sk_pred):.4f}')

assert n_diff == 0, 'ONNX cho kết quả khác scikit-learn — không được dùng bản xuất này.'
print('\n=> Hai bên cho kết quả giống hệt nhau. Bản ONNX dùng được.')

if KEEP_PROBABILITIES:
    max_gap = float(np.abs(np.asarray(onnx_out[1]) - model.predict_proba(X_test)).max())
    print(f'Sai lệch xác suất lớn nhất: {max_gap:.2e}  (do float32, không đáng kể)')

Đầu vào đưa cho ONNX (giá trị thô): min=0.00  max=7,661,226.50
So với dữ liệu đã chuẩn hoá       : min=0.0000  max=1.2000

Số mẫu kiểm tra      : 678
Số dự đoán lệch nhau : 0
Accuracy của ONNX    : 0.9941
Accuracy của sklearn : 0.9941

=> Hai bên cho kết quả giống hệt nhau. Bản ONNX dùng được.


In [26]:
# =========================================================
# 13.3 — Ghi lại hợp đồng vào/ra để bên triển khai dùng đúng
# =========================================================
io_spec = {
    'onnx_file': ONNX_PATH.name,
    'opset': OPSETS,
    'ir_version': int(onnx_model.ir_version),
    'operators': [n.op_type for n in onnx_model.graph.node],
    'input': {
        'name': 'input',
        'dtype': 'float32',
        'shape': ['N', len(FEATURES)],
        'feature_order': FEATURES,          # SAI THỨ TỰ = SAI KẾT QUẢ, không có cảnh báo
    },
    'output': {
        'name': 'output',
        'dtype': 'int64',
        'shape': ['N'],
        'meaning': {str(v): k for k, v in LABEL_MAP.items()},
    },
    'extra_output': ('probabilities [N,2] — cột 1 là xác suất scanport'
                     if KEEP_PROBABILITIES else None),
    'scaling': {
        'where': 'đã nhúng sẵn trong file ONNX (toán tử Mul + Add)',
        'method': SCALING['method'],
        'feature_range': SCALING['feature_range'],
        'action_required': 'KHÔNG — đưa thẳng giá trị thô vào, mô hình tự chuẩn hoá bên trong',
        'params_csv': 'scaler_params.csv (kèm theo, chỉ để tham khảo/tái lập)',
    },
    'preprocessing_required': {
        'note': ('Dữ liệu mới phải được làm sạch y hệt lúc huấn luyện. '
                 'Riêng phép chuẩn hoá thì KHÔNG cần làm — nó đã nằm trong mô hình.'),
        'steps': [
            'Bỏ nhóm cột định danh (IP, cổng, mốc thời gian) và các cột hằng số.',
            'Tốc độ của flow có duration = 0 hoặc vượt giới hạn vật lý 1 Gbps -> coi là thiếu.',
            'Giá trị IAT âm -> đặt về 0.',
            'Điền chỗ thiếu bằng đúng các giá trị trung vị dưới đây (học từ tập train).',
            'Đưa thẳng giá trị thô vào mô hình — KHÔNG tự chuẩn hoá lần nữa.',
        ],
        'impute_values': meta['imputation']['values'],
    },
    'verified': {'n_test_samples': int(len(sk_pred)), 'mismatches_vs_sklearn': n_diff},
}

(EXPORT_DIR / 'onnx_io_spec.json').write_text(
    json.dumps(io_spec, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Đã lưu: {EXPORT_DIR / "onnx_io_spec.json"}\n')

# scaler_params.csv do notebook tiền xử lý ghi thẳng vào đây, không cần chép lại
_sc = EXPORT_DIR / 'scaler_params.csv'
assert _sc.exists(), f'Thiếu {_sc} — chạy lại preprocessing_data_balance.ipynb'
print(f'Tham số chuẩn hoá     : {_sc.name} (đã có sẵn)')

print('\nCách gọi ở nơi khác (Python — các ngôn ngữ khác tương tự):')
print('''
    import numpy as np, onnxruntime as ort
    sess = ort.InferenceSession('decision_tree_model.onnx')
    X = np.asarray(rows, dtype=np.float32)      # [N, 24] GIA TRI THO, dung thu tu feature_order
    nhan = sess.run(None, {'input': X})[0]      # 0 = normal, 1 = scanport
''')
print('Không cần tự chuẩn hoá — phép co giãn về [0,1] đã nằm sẵn trong file ONNX.')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\model_export\onnx_io_spec.json

Tham số chuẩn hoá     : scaler_params.csv (đã có sẵn)

Cách gọi ở nơi khác (Python — các ngôn ngữ khác tương tự):

    import numpy as np, onnxruntime as ort
    sess = ort.InferenceSession('decision_tree_model.onnx')
    X = np.asarray(rows, dtype=np.float32)      # [N, 24] GIA TRI THO, dung thu tu feature_order
    nhan = sess.run(None, {'input': X})[0]      # 0 = normal, 1 = scanport

Không cần tự chuẩn hoá — phép co giãn về [0,1] đã nằm sẵn trong file ONNX.


In [27]:
# =========================================================
# 13.4 — Liệt kê nội dung hai thư mục kết quả
# =========================================================
DESC = {
    # --- model_export: những gì cần để chạy mô hình ở nơi khác ---
    'decision_tree_model.onnx':          'Mô hình ONNX — input:input, output:output',
    'onnx_io_spec.json':                 'Hợp đồng vào/ra + thứ tự cột bắt buộc',
    'scaler_params.csv':                 'Hai cột mean/scale (phép này đã nhúng sẵn trong ONNX)',
    'decision_tree_model.joblib':        'Mô hình cho Python/scikit-learn',
    # --- training_result: những gì cần để đánh giá và báo cáo ---
    'confusion_matrix.png':              'Ma trận nhầm lẫn — số lượng và tỉ lệ theo hàng',
    'confusion_matrix.csv':              'Ma trận nhầm lẫn dạng bảng',
    'confusion_matrix_normalized.csv':   'Ma trận nhầm lẫn đã chuẩn hoá theo hàng',
    'feature_importance.png':            '% ảnh hưởng của từng feature',
    'feature_importance.csv':            'Bảng đầy đủ, có cả cách đo bằng hoán vị',
    'feature_importance_comparison.png': 'Đối chiếu hai cách đo độ quan trọng',
    'classification_report.txt':         'Báo cáo phân loại dạng văn bản',
    'classification_report.csv':         'Báo cáo phân loại dạng bảng',
    'roc_pr_curves.png':                 'Đường ROC và Precision-Recall',
    'tuning_depth_curve.png':            'Ảnh hưởng của độ sâu tới học vẹt',
    'decision_tree.png':                 'Sơ đồ cây, 3 tầng đầu',
    'decision_tree_rules.txt':           'Toàn bộ luật của cây dạng văn bản',
    'model_comparison.csv':              'So sánh với mô hình cơ sở và Random Forest',
    'model_summary.json':                'Toàn bộ chỉ số và cấu hình',
}

def show(folder, tieu_de):
    files = sorted(folder.iterdir())
    total = sum(f.stat().st_size for f in files) / 1024
    print(f'{folder.name}/  —  {tieu_de}')
    for f in files:
        print(f'   {f.name:36s} {f.stat().st_size/1024:>8.1f} KB   {DESC.get(f.name, "")}')
    print(f'   {"":36s} {"":>8s}      {len(files)} file, {total:.1f} KB\n')
    return total

t1 = show(EXPORT_DIR, 'đem đi triển khai')
t2 = show(RESULT_DIR, 'đánh giá và báo cáo')
print(f'Tổng cộng: {(t1 + t2)/1024:.2f} MB')

model_export/  —  đem đi triển khai
   decision_tree_model.joblib                4.3 KB   Mô hình cho Python/scikit-learn
   decision_tree_model.onnx                  1.6 KB   Mô hình ONNX — input:input, output:output
   onnx_io_spec.json                         3.0 KB   Hợp đồng vào/ra + thứ tự cột bắt buộc
   scaler_params.csv                         0.7 KB   Hai cột mean/scale (phép này đã nhúng sẵn trong ONNX)
                                                      4 file, 9.6 KB

training_results/  —  đánh giá và báo cáo
   classification_report.csv                 0.2 KB   Báo cáo phân loại dạng bảng
   classification_report.txt                 0.5 KB   Báo cáo phân loại dạng văn bản
   confusion_matrix.csv                      0.1 KB   Ma trận nhầm lẫn dạng bảng
   confusion_matrix.png                     50.5 KB   Ma trận nhầm lẫn — số lượng và tỉ lệ theo hàng
   confusion_matrix_normalized.csv           0.1 KB   Ma trận nhầm lẫn đã chuẩn hoá theo hàng
   decision_tree.png       

---
## Tổng kết

### Kết quả

Xem `training_results/model_summary.json` để có toàn bộ số liệu. Các file quan trọng nhất:

**Trong `training_results/`** — để đánh giá:

- **`confusion_matrix.png`** — mô hình sai ở đâu, và sai theo kiểu nào
- **`feature_importance.png`** — feature nào quyết định kết quả, tính theo phần trăm
- **`decision_tree_rules.txt`** — luật đầy đủ, đọc được bằng mắt thường để kiểm chứng

**Trong `model_export/`** — để triển khai:

- **`decision_tree_model.onnx`** — chạy được ở C++/C#/Java, không cần Python
- **`onnx_io_spec.json`** — thứ tự 24 cột và các bước tiền xử lý bắt buộc. Thiếu file này thì
  mô hình vẫn chạy nhưng cho kết quả sai mà không báo lỗi

### Đọc kết quả cho đúng

Điểm số cao **không** có nghĩa mô hình đã sẵn sàng triển khai. Dữ liệu huấn luyện đến từ **một phiên
nmap duy nhất** kéo dài 5,4 phút, một kẻ tấn công, một mục tiêu, một cấu hình công cụ. Mô hình đã
chứng minh được nó nhận ra đúng phiên quét đó; nó **chưa** chứng minh được gì về khả năng nhận ra
port scan nói chung.

Cụ thể, mô hình hiện tại gần như chắc chắn sẽ bỏ sót:

- Quét chậm (`nmap -T2`, `-T1`) — nhịp gói hoàn toàn khác
- Quét UDP — không có cờ SYN/RST để bám vào
- Quét kiểu FIN / NULL / Xmas — chữ ký cờ khác hẳn
- Quét từ máy khác, tới mục tiêu khác

### Việc cần làm tiếp, theo thứ tự quan trọng

1. **Capture thêm 3–4 phiên quét đa dạng** rồi chạy lại toàn bộ ba notebook. Đây là việc duy nhất
   thực sự cải thiện được chất lượng mô hình ở thời điểm này.
2. **Capture dữ liệu brute-force** nếu vẫn giữ mục tiêu đa lớp như thiết kế ban đầu trong file Excel.
3. **Kiểm thử trên phiên capture hoàn toàn độc lập** — không phải tập test cắt ra từ cùng phiên, mà
   là một lần bắt gói khác hẳn. Đó mới là phép thử thật.
4. Sau khi có đủ dữ liệu, cân nhắc bổ sung **feature cross-flow** (số cổng đích khác nhau mà một
   nguồn chạm tới trong một cửa sổ thời gian). Quét cổng vốn là hiện tượng nhiều-flow; bộ feature
   hiện tại chỉ mô tả từng flow riêng lẻ nên không nắm được bản chất đó.